# PACE Phase 10 — Fair fixed-weight GA versus Pareto NSGA-II (analysis only)

**Purpose:** test Reviewer 1's multi-objective optimization concern. Runs a Pareto-based NSGA-II evolutionary search under the same 240-population / 45-generation / two-seed budget as the completed Phase 9 fixed-weight GA.

**Inputs:** (1) `PACE_Phase8_ESM2_results.zip` and (2) the uploaded verified `PACE_Phase9_Full_GA_Colab_Results (1).zip`. No raw CSV or original code ZIP is needed.

**Evidence controls:** exact Phase 9 archive SHA check, identical Phase 8 source hash, and numerical model-score consistency check before any algorithm comparison. If checks fail, **stop**: no scientific comparison should be reported.

**Note:** Only two seeds are included; results are pilot comparisons and need additional seeds before strong inference. Model predictions are not experimental potency or safety. No manuscript or supplementary edits.


In [ ]:
!pip -q install scikit-learn biopython requests numpy

In [ ]:
from pathlib import Path
Z=Path('/content/PACE_Phase8_ESM2_results.zip')
if not Z.is_file():
    from google.colab import files
    print('Upload only PACE_Phase8_ESM2_results.zip')
    uploaded=files.upload()
    assert 'PACE_Phase8_ESM2_results.zip' in uploaded, 'Upload the Phase 8 ESM-2 results ZIP'
    Z.write_bytes(uploaded['PACE_Phase8_ESM2_results.zip'])
else:
    print('Phase 8 results already present; no upload needed', Z.stat().st_size)

In [ ]:
from pathlib import Path
from google.colab import files
p = list(Path('/content').glob('PACE_Phase9_Full_GA_Colab_Results*.zip'))
if not p:
    print('Upload your completed Phase 9 results ZIP (the file you sent to ChatGPT):')
    files.upload()
    p = list(Path('/content').glob('PACE_Phase9_Full_GA_Colab_Results*.zip'))
assert p, 'Phase 9 result ZIP must be uploaded'
print('Phase 9 results found:', [str(x) for x in p])


In [ ]:
# Fit original-method predictor suite once. The namespace persists in memory.
# If GA fails, do NOT rerun this cell; rerun the next GA cell only.
PACE_CTX = globals().get('PACE_CTX', {'__name__': 'pace_phase9_models'})
if PACE_CTX.get('_models_ready'):
    print('Models are already fitted in this Colab session; skipping retrain.')
else:
    RUNNER_HEADER = '"""PACE Phase 9 — Fully refitted GA ablations from verified Phase 8 source output.\nNo manuscript or original author file edits. Quantitative MIC/HC50 labels and 90D features\ncome from user\'s ESM2 Phase 8 archive, and original FASTA labels from fixed-source GitHub.\n"""\nimport io, csv, json, time, zipfile, hashlib, collections, sys, importlib.util\nfrom pathlib import Path\nimport numpy as np\nfrom sklearn.ensemble import (ExtraTreesClassifier,RandomForestClassifier,\n    HistGradientBoostingClassifier,ExtraTreesRegressor)\nfrom sklearn.svm import SVC\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\nimport requests\nROOT=Path(\'/content\');OUT=ROOT/\'pace_phase9_colab\';OUT.mkdir(exist_ok=True)\nZ=ROOT/\'PACE_Phase8_ESM2_results.zip\'\nassert Z.exists(), f\'Missing {Z}; upload Phase8 ESM2 results ZIP first\'\nwith zipfile.ZipFile(Z) as z:\n    manifest=json.loads(z.read(\'manifest.json\'))\n    db=np.load(io.BytesIO(z.read(\'all_embeddings_and_features.npz\')))\n    seq_ref=np.asarray(db[\'sequences\']);X_ref=np.asarray(db[\'X90\'])\n    predtab=z.read(\'quantitative_predictions.csv\').decode()\n    (OUT/\'features_original_source.py\').write_bytes(z.read(\'pace_original_features.py\'))\nnpz_dict={s:i for i,s in enumerate(seq_ref)}\nsha=hashlib.sha256()\nsource={}\nfor name,info in manifest[\'source_files\'].items():\n    print(\'Downloading\',name,flush=True)\n    r=requests.get(info[\'url\'],timeout=120);r.raise_for_status()\n    assert hashlib.sha256(r.content).hexdigest()==info[\'sha256\'],f\'Hash mismatch {name}\'\n    source[name]=r.content.decode(\'utf-8-sig\')\n(OUT/\'input_source_manifest.json\').write_text(json.dumps(manifest,indent=2))\nspec=importlib.util.spec_from_file_location(\'pace_features\',OUT/\'features_original_source.py\')\nfeatures=importlib.util.module_from_spec(spec);spec.loader.exec_module(features)\nALPHABET_CANON=set(\'ACDEFGHIKLMNPQRSTVWY\')\ndef parse_fasta(txt):\n    seqs=[];buf=[]\n    for ln in txt.splitlines():\n        if ln.startswith(\'>\'):\n            if buf:seqs.append(\'\'.join(buf));buf=[]\n        else:buf.append(ln.strip())\n    if buf:seqs.append(\'\'.join(buf))\n    return seqs\ndef clean_many(raw):\n    r=[]\n    for x in parse_fasta(raw):\n        s=features.clean(x)\n        if s is not None and 5<=len(s)<=60:r.append(s)\n    return r\ntrainPos=clean_many(source[\'AMP_train\']);trainNeg=clean_many(source[\'nonAMP_train\'])\ntestPos=clean_many(source[\'AMP_test\']);testNeg=clean_many(source[\'nonAMP_test\'])\nextPos=clean_many(source[\'external_AMP\']);extNeg=clean_many(source[\'external_nonAMP\'])\ndef dedup(pos,neg):\n    d={}\n    for s in pos:d.setdefault(s,1)\n    for s in neg:d.setdefault(s,0)\n    return d\ntrain=dedup(trainPos,trainNeg);test={s:y for s,y in dedup(testPos,testNeg).items() if s not in train}\nseen=set(train)|set(test)\nexternal={s:y for s,y in dedup(extPos,extNeg).items() if s not in seen}\nassert (len(train),sum(train.values()),len(test),sum(test.values()),len(external),sum(external.values()))==(6271,3124,1568,796,1848,671)\ncl={**train,**test}; ic=np.array([npz_dict[s] for s in cl]);yc=np.array(list(cl.values()))\nprint(\'Fitting original PACE four-member soft-voting classifier on 7839 train+heldout peptides\',flush=True)\ncls_models={\n    \'GB\':HistGradientBoostingClassifier(max_iter=400,learning_rate=.07,random_state=2024),\n    \'ET\':ExtraTreesClassifier(n_estimators=400,random_state=2024,n_jobs=4),\n    \'SVM\':Pipeline([(\'s\',StandardScaler()),(\'m\',SVC(C=4,gamma=\'scale\',probability=True,random_state=2024))]),\n    \'RF\':RandomForestClassifier(n_estimators=400,random_state=2024,n_jobs=4)\n}\nfor name,model in cls_models.items():\n    model.fit(X_ref[ic],yc);print(\'classifier fitted\',name,flush=True)\nqs=[r for r in csv.DictReader(io.StringIO(predtab)) if r[\'representation\']==\'90D_PACE\']\nquant={}\nfor r in qs:quant.setdefault(r[\'endpoint\'],{}).setdefault(r[\'sequence\'],float(r[\'observed_log10\']))\nexp={\'E. faecium\':780,\'S. aureus\':4128,\'K. pneumoniae\':743,\'A. baumannii\':306,\'P. aeruginosa\':2517,\'E. cloacae\':304,\'HC50\':1271}\nassert all(len(quant[x])==n for x,n in exp.items()),{x:len(quant.get(x,{})) for x in exp}\nreg={}\nfor target in list(exp)[:-1]:\n    q=quant[target];ix=np.array([npz_dict[s] for s in q]);y=np.array(list(q.values()))\n    m=ExtraTreesRegressor(n_estimators=300,random_state=2024,n_jobs=4);m.fit(X_ref[ix],y);reg[target]=m\n    print(\'MIC fitted\',target,len(y),flush=True)\nq=quant[\'HC50\'];ix=np.array([npz_dict[s] for s in q]);y=np.array(list(q.values()))\nhem=ExtraTreesRegressor(n_estimators=300,random_state=2024,n_jobs=4);hem.fit(X_ref[ix],y)\nprint(\'HC50 fitted\',len(y),flush=True)\nPOOL=sorted(set(s for s,y in {**train,**test,**external}.items() if y==1)|set().union(*(set(q.keys()) for q in quant.values())))\nK=3; inv=collections.defaultdict(list);sets=[]\nfor i,s in enumerate(POOL):\n    a=set(s[j:j+3] for j in range(len(s)-2));sets.append(a)\n    for k in a:inv[k].append(i)\ndef maximum_similarity(s):\n    a={s[j:j+3] for j in range(len(s)-2)};cnt=collections.Counter()\n    for k in a:\n        for i in inv.get(k,()):cnt[i]+=1\n    best=0.\n    for i,shared in cnt.items():\n        j=shared/(len(a)+len(sets[i])-shared)\n        if j>best:best=j\n    return best\ncache={};counts={\'features\':0}\ndef get_scores(seqs):\n    new=[s for s in dict.fromkeys(seqs) if s not in cache]\n    if new:\n        X=features.featurize_many(new)\n        p=np.mean([m.predict_proba(X)[:,1] for m in cls_models.values()],axis=0)\n        mm=np.column_stack([reg[t].predict(X) for t in reg]);hc=hem.predict(X)\n        for i,s in enumerate(new):\n            sim=maximum_similarity(s);mean=float(mm[i].mean());worst=float(mm[i].max());ti=float(hc[i]-mean);charge=features.net_charge(s)\n            cache[s]={\'p_amp\':float(p[i]),\'mic_mean\':mean,\'mic_worst\':worst,\'hc50\':float(hc[i]),\'ti\':ti,\'sim\':sim,\'charge\':float(charge),\'length\':len(s),\'sequence\':s}\n        counts[\'features\']+=len(new)\n    return [cache[s] for s in seqs]\nprovenance={\'source\':\'Analysis-only independent refit of PACE 90D models from Phase8 curated labels and pinned FASTA sources, reconstructed from original published code\',\n  \'quantitative_labels_source\':\'Phase8 quantitative_predictions 90D_PACE observed_log10 (uniquely deduplicated by sequence and endpoint)\',\n  \'classifier_training_n\':len(cl),\'classifier_names\':list(cls_models),\'classifier_trees\':400,\'regressor_trees\':300,\n  \'novelty_known_peptides\':len(POOL),\'ga_seeds\':[7,29],\'original_pace_source\':\'PACE_reproducibility_code.zip\',\n  \'original_pace_code_zip_sha256\':\'8a72718087b551b968ed0c85652ae40dda029ae61a5dc7c7cbff87184eceace7\',\n  \'limits\':\'Newly fitted models may not bitwise reproduce prior author weights; same descriptor code and model settings\',\n  \'phase8_sha256\':hashlib.sha256(Z.read_bytes()).hexdigest(),\n  \'note\':\'Original PACE ZIP is NOT required to run: archived code SHA is documented. GA ablations and diversity selection are an extension of original workflow.\'}\n(OUT/\'full_protocol_model_provenance.json\').write_text(json.dumps(provenance,indent=2))\nALPHABET=list(\'ADEFGHIKLNPQRSTVWY\');LMIN=12;LMAX=30;POP=240;GENS=45;ELITE=24\nTARGETS=list(reg)\ndef fitness(v,arm):\n    a=2*v[\'p_amp\']-v[\'mic_mean\'];b=.5*v[\'mic_worst\'];c=.6*float(np.clip(v[\'ti\'],-2,4));d=2*max(0,v[\'sim\']-.55);e=.35*max(0,abs(v[\'charge\']-6)-3)\n    if arm==\'full\':return a-b+c-d-e\n    if arm==\'no_hemolysis\':return a-b-d-e\n    if arm==\'no_novelty\':return a-b+c-e\n    if arm==\'no_worst_case\':return a+c-d-e\n    if arm==\'potency_recognition_only\':return a-e\n    raise KeyError(arm)\n\ndef random_seq(rng):return \'\'.join(rng.choice(ALPHABET,int(rng.integers(LMIN,LMAX+1))))\ndef mutate(s,rng):\n    s=list(s);op=rng.random()\n    if op<.70 or len(s)<=LMIN:\n        for _ in range(int(rng.integers(1,4))):s[int(rng.integers(len(s)))]=str(rng.choice(ALPHABET))\n    elif op<.85 and len(s)<LMAX:s.insert(int(rng.integers(len(s)+1)),str(rng.choice(ALPHABET)))\n    else:del s[int(rng.integers(len(s)))]\n    return \'\'.join(s)\ndef crossover(a,b,rng):\n    s=a[:int(rng.integers(1,len(a)))]+b[int(rng.integers(1,len(b))):]\n    if len(s)>LMAX:s=s[:LMAX]\n    while len(s)<LMIN:s+=str(rng.choice(ALPHABET))\n    return s\n\ndef pair_sim(a,b):\n    u=set(a[i:i+3] for i in range(len(a)-2));v=set(b[i:i+3] for i in range(len(b)-2))\n    return len(u&v)/len(u|v)\n\ndef select_leads(unique,arm):\n    order=sorted(unique,key=lambda s:fitness(cache[s],arm),reverse=True)\n    passed=[];rejected_by=collections.Counter()\n    for s in order:\n        v=cache[s]\n        if v[\'p_amp\']<.9:rejected_by[\'p_amp\']+=1;continue\n        if v[\'mic_worst\']>np.log10(32):rejected_by[\'mic\']+=1;continue\n        if v[\'ti\']<1:rejected_by[\'selectivity\']+=1;continue\n        if v[\'sim\']>=.55:rejected_by[\'novelty\']+=1;continue\n        passed.append(s)\n    leads=[]\n    for s in passed:\n        if all(pair_sim(s,t)<=.45 for t in leads):leads.append(s)\n        if len(leads)==20:break\n    return leads,len(passed),dict(rejected_by)\n\ndef run(arm,seed):\n    rng=np.random.default_rng(seed)\n    pop=[random_seq(rng) for _ in range(POP)];seen=set();history=[]\n    for g in range(1,GENS+1):\n        rec=get_scores(pop);seen.update(pop)\n        fs=[fitness(s,arm) for s in rec]\n        order=np.argsort(-np.array(fs)); pop=[pop[j] for j in order];sc=[rec[j] for j in order];fs=[fs[j] for j in order]\n        history.append({\'arm\':arm,\'seed\':seed,\'gen\':g,\'best_native_fitness\':fs[0],\'population_mean_p_amp\':float(np.mean([s[\'p_amp\'] for s in sc])),\n          \'population_mean_mic_log10\':float(np.mean([s[\'mic_mean\'] for s in sc])),\'population_mean_ti_log10\':float(np.mean([s[\'ti\'] for s in sc])),\n          \'mean_length\':float(np.mean([s[\'length\'] for s in sc])),\'mean_novelty_similarity\':float(np.mean([s[\'sim\'] for s in sc]))})\n        if g==GENS:break\n        nxt=pop[:ELITE]\n        while len(nxt)<POP:\n            def tournament():\n                ids=rng.integers(0,len(pop),3)\n                return pop[int(max(ids,key=lambda i:fs[i]))]\n            c=crossover(tournament(),tournament(),rng) if rng.random()<.6 else tournament()\n            if rng.random()<.9:c=mutate(c,rng)\n            if LMIN<=len(c)<=LMAX:nxt.append(c)\n        pop=nxt\n    leads,feasible,rej=select_leads(seen,arm)\n    values=[cache[s] for s in leads]\n    out={\'arm\':arm,\'seed\':seed,\'unique_scored\':len(seen),\'feasible_hard_filters\':feasible,\'n_diverse_leads\':len(leads),\n         \'best_native_fitness\':max(fitness(cache[s],arm) for s in seen),\n         \'fraction_known_sim_above_0_55\':np.mean([cache[s][\'sim\']>.55 for s in seen]),\n         \'fraction_length_29_30_all\':np.mean([cache[s][\'length\']>=29 for s in seen]),\n         \'lead_avg_length\':float(np.mean([v[\'length\'] for v in values])) if values else None,\n         \'lead_avg_mic_log10\':float(np.mean([v[\'mic_mean\'] for v in values])) if values else None,\n         \'lead_avg_worst_mic_log10\':float(np.mean([v[\'mic_worst\'] for v in values])) if values else None,\n         \'lead_avg_ti_log10\':float(np.mean([v[\'ti\'] for v in values])) if values else None,\n         \'lead_avg_p_amp\':float(np.mean([v[\'p_amp\'] for v in values])) if values else None,\n         \'lead_avg_known_sim\':float(np.mean([v[\'sim\'] for v in values])) if values else None,\n         \'lead_avg_full_fitness\':float(np.mean([fitness(v,\'full\') for v in values])) if values else None,\n         \'filter_reject_reasons\':rej}\n    print(\'RUN\',json.dumps({k:v for k,v in out.items() if k!=\'filter_reject_reasons\'}),flush=True)\n    return out,history,[{\'arm\':arm,\'seed\':seed,\'rank\':i+1,**cache[s], \'full_fitness\':fitness(cache[s],\'full\'), \'native_fitness\':fitness(cache[s],arm)} for i,s in enumerate(leads)],seen\n\ndef save_table(path,rows):\n    if not rows:return\n    cols=list(rows[0]);\n    with path.open(\'w\',newline=\'\') as f:\n        w=csv.DictWriter(f,cols,extrasaction=\'ignore\');w.writeheader();w.writerows(rows)\n\n'
    exec(compile(RUNNER_HEADER,'PACE_Phase9_Full_GA_FIXED_Runner.py','exec'), PACE_CTX)
    assert all(k in PACE_CTX for k in ['cls_models','reg','hem','get_scores','features']), 'Missing fitted models'
    PACE_CTX['_models_ready']=True
    print('ALL TRAINED MODELS READY. Next: run GA cell.')

In [ ]:
# The following cell uses saved fitted models if the earlier Phase 9 notebook
# is still in the same runtime. Otherwise the preceding cell refits them.
from pathlib import Path
assert PACE_CTX.get('_models_ready'), 'Run model-fitting cell first'
NSGA2_SOURCE = '"""PACE Phase 10: analysis-only Pareto NSGA-II against Phase 9 fixed-weight GA.\nRun with PACE_CTX (Phase 9 model-fitting context), not as a standalone script.\nNo manuscript or supplementary file edits.\n"""\nimport time, json, zipfile, hashlib, csv\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom itertools import combinations\n\nROOT10 = Path(\'/content\')\nOUT10 = ROOT10 / \'pace_phase10_colab\'\nOUT10.mkdir(exist_ok=True)\nphase9_candidates = sorted(ROOT10.glob(\'PACE_Phase9_Full_GA_Colab_Results*.zip\'))\nEXPECTED_PHASE9_SHA256 = \'09cd801741bdaca6fb469cf082beefa8eadcf26297fea64ce5f08228b766f350\'\nphase9_path = next((p for p in phase9_candidates if hashlib.sha256(p.read_bytes()).hexdigest()==EXPECTED_PHASE9_SHA256), None)\nassert phase9_path is not None, \'The previously independently checked Phase 9 ZIP was not found. Please upload PACE_Phase9_Full_GA_Colab_Results (1).zip without modification.\'\nwith zipfile.ZipFile(phase9_path) as z9:\n    prior = pd.read_csv(z9.open(\'phase9_ablation_leads.csv\'))\n    prior_runs = pd.read_csv(z9.open(\'phase9_ablation_run_summary.csv\'))\n    prior_provenance = json.loads(z9.read(\'full_protocol_model_provenance.json\'))\nassert prior_provenance[\'phase8_sha256\'] == hashlib.sha256((ROOT10/\'PACE_Phase8_ESM2_results.zip\').read_bytes()).hexdigest(), \'Phase8 source archive mismatch with Phase9 baseline\'\n\n# Mandatory prediction-consistency check: compare prior PACE scores under the refitted models.\nref=prior[(prior.arm==\'full\')&(prior.seed==7)].head(10)\nsc=get_scores(ref.sequence.tolist())\nfields=(\'p_amp\',\'mic_mean\',\'mic_worst\',\'hc50\',\'ti\',\'sim\',\'charge\')\ndiffs={k: float(np.max(np.abs(np.asarray([x[k] for x in sc])-ref[k].to_numpy(float)))) for k in fields}\n(OUT10/\'baseline_model_consistency.json\').write_text(json.dumps(diffs,indent=2))\nprint(\'Phase9 model score differences:\', diffs, flush=True)\nassert all(v<1e-5 for v in diffs.values()), \'Model predictions are NOT the same as Phase 9! Do not compare. Refit models in identical software environment or rerun both algorithms together.\'\n\n# Pareto objectives: minimize [-probability AMP, average log10 MIC, worst log10 MIC,\n# -log10 selectivity, known-3-mer similarity, soft charge excess].\n# Hard feasibility: P(AMP)>=.9, worst MIC<=32uM, log10 selectivity>=1, sim<.55.\nOBJECTIVE_NAMES = (\'negative_p_amp\',\'mean_log10_mic\',\'worst_log10_mic\',\'negative_log10_selectivity\',\'known_3mer_similarity\',\'charge_penalty\')\nTOL=1e-12\n\ndef objectives(records):\n    return np.asarray([[-v[\'p_amp\'], v[\'mic_mean\'], v[\'mic_worst\'], -v[\'ti\'], v[\'sim\'], max(0.,abs(v[\'charge\']-6.)-3.)] for v in records], dtype=float)\n\ndef violations(records):\n    return np.asarray([max(0.,.9-v[\'p_amp\']) + max(0.,v[\'mic_worst\']-np.log10(32.)) + max(0.,1.-v[\'ti\']) + max(0.,v[\'sim\']-.55) for v in records], dtype=float)\n\ndef ranks_and_crowding(records):\n    X=objectives(records);cv=violations(records);n=len(records)\n    # Deb constraint-domination: feasible > infeasible; otherwise smaller total CV;\n    # feasible-vs-feasible uses Pareto objective dominance.\n    feasible=cv<=TOL\n    ordinary=(np.all(X[:,None,:] <= X[None,:,:] + TOL, axis=2) & np.any(X[:,None,:] < X[None,:,:]-TOL,axis=2))\n    dom=np.where(feasible[:,None] & ~feasible[None,:], True,\n                 np.where(~feasible[:,None] & feasible[None,:], False,\n                          np.where(~feasible[:,None] & ~feasible[None,:],cv[:,None] < cv[None,:]-TOL,ordinary)))\n    np.fill_diagonal(dom,False)\n    rank=np.full(n,-1,dtype=int);remain=np.ones(n,dtype=bool);fronts=[];k=0\n    while np.any(remain):\n        indices=np.where(remain)[0]\n        counts=dom[np.ix_(indices,indices)].sum(axis=0)\n        front=indices[counts==0]\n        assert len(front),\'Dominance graph cycle; check constraints/comparisons\'\n        rank[front]=k;remain[front]=False;fronts.append(front);k+=1\n    crowd=np.zeros(n,dtype=float)\n    for front in fronts:\n        if len(front)<=2:\n            crowd[front]=np.inf;continue\n        crowd[front]=0\n        for j in range(X.shape[1]):\n            order=front[np.argsort(X[front,j],kind=\'stable\')]\n            gap=X[order[-1],j]-X[order[0],j]\n            if gap>TOL:\n                crowd[order[0]]=np.inf;crowd[order[-1]]=np.inf\n                crowd[order[1:-1]]+=(X[order[2:],j]-X[order[:-2],j])/gap\n    return rank,crowd,fronts\n\ndef nsga_select(pop, n=240):\n    # Avoid redundant strings in the Pareto population so duplicate genotypes\n    # cannot crowd out different candidates with equivalent scores.\n    pop=list(dict.fromkeys(pop))\n    records=get_scores(pop)\n    rank,crowd,fronts=ranks_and_crowding(records)\n    out=[]\n    for front in fronts:\n        if len(out)+len(front)<=n:out.extend(front.tolist())\n        else:\n            order=front[np.argsort(-crowd[front],kind=\'stable\')]\n            out.extend(order[:n-len(out)].tolist());break\n    return [pop[i] for i in out]\n\ndef parent_tournament(pop,rank,crowd,rng):\n    i,j=rng.integers(0,len(pop),2)\n    if rank[i]!=rank[j]:return pop[i if rank[i]<rank[j] else j]\n    if crowd[i]!=crowd[j]:return pop[i if crowd[i]>crowd[j] else j]\n    return pop[int(i if rng.random()<.5 else j)]\n\ndef comparable_crowding_sample(pop):\n    records=get_scores(pop);rank,crowd,fronts=ranks_and_crowding(records)\n    # Count feasible and feasible first-front members in the evolving population.\n    v=violations(records);idx=np.where(v<=TOL)[0]\n    return rank,crowd,{\'feasible_population\':len(idx),\'pareto_front_population\':len(fronts[0]),\'feasible_front_population\':sum(v[i]<=TOL for i in fronts[0])}\n\ndef feasible(s):\n    v=cache[s]\n    return v[\'p_amp\']>=.9 and v[\'mic_worst\']<=np.log10(32.) and v[\'ti\']>=1 and v[\'sim\']<.55\n\ndef parse_pareto_20(pop,archive):\n    # First prioritize feasible final-generation Pareto individuals by crowding.\n    pf=[s for s in dict.fromkeys(pop) if feasible(s)]\n    leads=[]\n    if pf:\n        rec=get_scores(pf);rk,cr,_=ranks_and_crowding(rec)\n        order=sorted(range(len(pf)),key=lambda i:(rk[i],-cr[i],-fitness(cache[pf[i]],\'full\')))\n        for i in order:\n            s=pf[i]\n            if all(pair_sim(s,t)<=.45 for t in leads): leads.append(s)\n            if len(leads)==20:break\n    # If needed, explicitly track a scalar-rank fallback to achieve a 20-lead library.\n    fallback=0\n    if len(leads)<20:\n        for s in sorted(archive,key=lambda t:fitness(cache[t],\'full\'),reverse=True):\n            if not feasible(s) or s in leads:continue\n            if all(pair_sim(s,t)<=.45 for t in leads):leads.append(s);fallback+=1\n            if len(leads)==20:break\n    return leads,fallback\n\ndef execute_nsga(seed):\n    rng=np.random.default_rng(seed)\n    pop=[random_seq(rng) for _ in range(240)]\n    seen=set();history=[]\n    for gen in range(1,46):\n        vals=get_scores(pop);seen.update(pop)\n        rank,crowd,counts=comparable_crowding_sample(pop)\n        history.append({\'seed\':seed,\'generation\':gen,\'unique_seen\':len(seen),**counts,\n                        \'mean_p_amp\':float(np.mean([x[\'p_amp\'] for x in vals])),\n                        \'mean_log10_mic\':float(np.mean([x[\'mic_mean\'] for x in vals])),\n                        \'mean_log10_selectivity\':float(np.mean([x[\'ti\'] for x in vals]))})\n        if gen==45:break\n        children=[]\n        while len(children)<240:\n            p1=parent_tournament(pop,rank,crowd,rng)\n            s=crossover(p1,parent_tournament(pop,rank,crowd,rng),rng) if rng.random()<.6 else p1\n            if rng.random()<.9:s=mutate(s,rng)\n            if 12<=len(s)<=30:children.append(s)\n        seen.update(children)\n        # Standard elitist NSGA-II environmental selection: parent + offspring,\n        # non-dominated ranks, then crowding within the final partial front.\n        pop=nsga_select(pop+children,240)\n        if len(pop)<240:\n            while len(pop)<240:\n                s=random_seq(rng)\n                if s not in pop:pop.append(s);seen.add(s)\n        if gen%10==0:print(\'NSGA-II\',seed,\'completed generation\',gen,\'unique\',len(seen),flush=True)\n    chosen,fallback=parse_pareto_20(pop,seen)\n    assert len(chosen)==20, f\'Found only {len(chosen)} distinct feasible leads for seed {seed}\'\n    rows=[]\n    for k,s in enumerate(chosen,1):\n        v=cache[s]\n        rows.append({\'algorithm\':\'NSGA-II\',\'seed\':seed,\'rank\':k,**v,\'predicted_geom_mean_mic_um\':10**v[\'mic_mean\'],\n                     \'predicted_worst_mic_um\':10**v[\'mic_worst\'],\'predicted_selectivity_index\':10**v[\'ti\'],\n                     \'full_fitness\':fitness(v,\'full\')})\n    return rows,history, {\'seed\':seed,\'unique_evaluated\':len(seen),\'pareto_population_selection\':20-fallback,\n                          \'scalar_fallback_selection\':fallback,\'final_feasible_in_population\':sum(feasible(s) for s in pop)}\n\nif __name__==\'__main__\':\n    t=time.time();all_leads=[];all_hist=[];run_summary=[]\n    for seed in (7,29):\n        lead_rows,hist_rows,stat=execute_nsga(seed)\n        all_leads.extend(lead_rows);all_hist.extend(hist_rows);run_summary.append(stat)\n        print(\'FINISHED NSGA-II SEED\',seed,stat,flush=True)\n    nsga=pd.DataFrame(all_leads)\n    nsga.to_csv(OUT10/\'nsga2_selected_leads.csv\',index=False)\n    pd.DataFrame(all_hist).to_csv(OUT10/\'nsga2_generation_history.csv\',index=False)\n    pd.DataFrame(run_summary).to_csv(OUT10/\'nsga2_run_summary.csv\',index=False)\n    base=prior[(prior.arm==\'full\') & (prior.seed.isin([7,29]))].copy()\n    base[\'predicted_geom_mean_mic_um\']=10**base[\'mic_mean\']\n    base[\'predicted_worst_mic_um\']=10**base[\'mic_worst\']\n    base[\'predicted_selectivity_index\']=10**base[\'ti\']\n    base[\'algorithm\']=\'Fixed-weight GA\'\n    cols=[\'algorithm\',\'seed\',\'rank\',\'p_amp\',\'mic_mean\',\'mic_worst\',\'hc50\',\'ti\',\'sim\',\'charge\',\'length\',\'sequence\',\n          \'predicted_geom_mean_mic_um\',\'predicted_worst_mic_um\',\'predicted_selectivity_index\',\'full_fitness\']\n    combined=pd.concat([base[cols],nsga[cols]],ignore_index=True)\n    combined.to_csv(OUT10/\'matched_budget_algorithm_comparison_leads.csv\',index=False)\n    metric=combined.groupby([\'algorithm\',\'seed\'],as_index=False)[[\'predicted_geom_mean_mic_um\',\'predicted_worst_mic_um\',\n               \'predicted_selectivity_index\',\'p_amp\',\'sim\',\'length\']].mean()\n    metric.to_csv(OUT10/\'matched_budget_algorithm_comparison_summary.csv\',index=False)\n    meta={\'source_phase9_archive\':phase9_path.name,\'source_phase9_sha256\':hashlib.sha256(phase9_path.read_bytes()).hexdigest(),\n         \'source_phase8_sha256\':prior_provenance[\'phase8_sha256\'],\'model_consistency_max_absolute_differences\':diffs,\n         \'nsga_objectives\':OBJECTIVE_NAMES,\'constraints\':\'PAMP>=0.9, worstMIC<=32uM, log10SI>=1, sim<0.55; pairwise diversity<=0.45 at lead selection\',\n         \'seeds\':[7,29],\'population\':240,\'generations\':45,\'method\':\'Deb-style constraint dominance, elitist nondominated sorting and crowding distance; common crossover and mutation operators\',\n         \'note\':\'Analysis-only matched-budget benchmark. NSGA-II optimizes a genuine Pareto objective vector; fixed-weight GA baseline comes from uploaded Phase9 artifact; resulting selectivity and MIC are modeled, not measured.\',\n         \'runtime_seconds\':time.time()-t}\n    (OUT10/\'run_metadata.json\').write_text(json.dumps(meta,indent=2))\n    out=ROOT10/\'PACE_Phase10_NSGA2_Results.zip\'\n    with zipfile.ZipFile(out,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in OUT10.glob(\'*\'):\n            if p.is_file():z.write(p,arcname=p.name)\n    print(\'COMPLETED Phase 10:\',out,\'bytes\',out.stat().st_size,flush=True)\n'
Path('/content/pace_phase10_colab').mkdir(exist_ok=True)
Path('/content/pace_phase10_colab/PACE_Phase10_NSGA2_Runner.py').write_text(NSGA2_SOURCE)
PACE_CTX['__name__'] = '__main__'
exec(compile(NSGA2_SOURCE, 'PACE_Phase10_NSGA2_Runner.py', 'exec'), PACE_CTX)
assert Path('/content/PACE_Phase10_NSGA2_Results.zip').exists(), 'Result ZIP not created'


In [ ]:
from google.colab import files
p='/content/PACE_Phase10_NSGA2_Results.zip'
assert Path(p).is_file(), 'Run NSGA-II experiment first'
files.download(p)


## After the run

Upload **`PACE_Phase10_NSGA2_Results.zip`** here for independent checking. Preserve your previous Phase 9 output as-is. Comparison will cover predicted MIC/HC50, selectivity, Pareto fronts, feasible leads, diversity, and differences by seed. The notebook's safety check may stop if model-score hashes or predictions differ; in that case share the error rather than modifying or guessing values.
